# 1. Setup & Google Drive Integration
Installs dependencies and mounts your Google Drive to save checkpoints so you don't lose progress.

In [ ]:
!pip install datasets torch tqdm

import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Create a directory for our checkpoints
CHECKPOINT_DIR = '/content/drive/MyDrive/NNUE_Checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoints will be saved to: {CHECKPOINT_DIR}")

# 2. Ultra-Fast FEN Parser
Bypasses the slow `python-chess` library by mapping FEN strings directly to tensors using Python dictionaries and list comprehensions.

In [ ]:
import torch
import math

# Map characters to (color, piece_type) -> (1 for White, 0 for Black), (0=P,1=N,2=B,3=R,4=Q,5=K)
PIECE_MAP = {
    'P': (1, 0), 'N': (1, 1), 'B': (1, 2), 'R': (1, 3), 'Q': (1, 4), 'K': (1, 5),
    'p': (0, 0), 'n': (0, 1), 'b': (0, 2), 'r': (0, 3), 'q': (0, 4), 'k': (0, 5)
}

def fast_fen_to_features(fen):
    # Split FEN to get just the board layout
    board_part = fen.split(' ')[0]
    
    # Initialize empty tensor (768 features)
    features = torch.zeros(768, dtype=torch.float32)
    
    sq = 56 # Top-left is A8 (index 56)
    for char in board_part:
        if char == '/':
            sq -= 16 # Move down a rank
        elif char.isdigit():
            sq += int(char)
        else:
            c, pt = PIECE_MAP[char]
            idx = (c * 6 + pt) * 64 + sq
            features[idx] = 1.0
            sq += 1
            
    return features


# 3. PyTorch DataLoader
Creates a fast streaming dataset.

In [ ]:
from torch.utils.data import IterableDataset, DataLoader
from datasets import load_dataset

class FastChessDataset(IterableDataset):
    def __init__(self, hf_dataset):
        self.dataset = hf_dataset
        
    def __iter__(self):
        for row in self.dataset:
            fen = row['fen']
            cp = row['cp']
            mate = row['mate']
            
            # Calculate WDL
            if mate is not None:
                wdl = 1.0 if mate > 0 else 0.0
            elif cp is not None:
                wdl = 1.0 / (1.0 + math.pow(10.0, -cp / 400.0))
            else:
                continue # Skip invalid rows
                
            features = fast_fen_to_features(fen)
            yield features, torch.tensor([wdl], dtype=torch.float32)

print("Loading dataset stream...")
hf_dataset = load_dataset("Lichess/chess-position-evaluations", split="train", streaming=True)
chess_dataset = FastChessDataset(hf_dataset)

BATCH_SIZE = 16384 # Doubled batch size thanks to AMP
dataloader = DataLoader(chess_dataset, batch_size=BATCH_SIZE, num_workers=0, pin_memory=True)


# 4. NNUE Architecture


In [ ]:
import torch.nn as nn

class NNUE(nn.Module):
    def __init__(self):
        super(NNUE, self).__init__()
        self.fc1 = nn.Linear(768, 256)
        self.fc2 = nn.Linear(256, 32)
        self.fc3 = nn.Linear(32, 32)
        self.fc4 = nn.Linear(32, 1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        x = self.sigmoid(self.fc4(x))
        return x


# 5. Training Loop with AMP & Checkpointing
Automatically saves progress to Drive and resumes if Colab crashes.

In [ ]:
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

model = NNUE().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scaler = torch.amp.GradScaler('cuda') # For Mixed Precision

TARGET_BATCHES = 100000 # Let it run for a long time!
SAVE_INTERVAL = 1000   # Save to Drive every 1000 batches

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TARGET_BATCHES)
criterion = nn.MSELoss()

CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, 'nnue_checkpoint.pth')
start_batch = 0

# Load Checkpoint if it exists
if os.path.exists(CHECKPOINT_PATH):
    print("Found existing checkpoint! Resuming...")
    checkpoint = torch.load(CHECKPOINT_PATH, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scaler.load_state_dict(checkpoint['scaler_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    start_batch = checkpoint['batch_idx']
    print(f"Resumed from batch {start_batch}")
    
    # Skip data if resuming
    if start_batch > 0:
        print(f"Skipping {start_batch} batches to resume position...")
        skip_iter = iter(dataloader)
        for _ in tqdm(range(start_batch), desc="Skipping"):
            try:
                next(skip_iter)
            except StopIteration:
                break
        dataloader_iter = skip_iter
    else:
        dataloader_iter = iter(dataloader)
else:
    dataloader_iter = iter(dataloader)

model.train()
progress_bar = tqdm(total=TARGET_BATCHES, initial=start_batch, desc="Training Batches")

batch_idx = start_batch

for X_batch, y_batch in dataloader_iter:
    X_batch = X_batch.to(device, non_blocking=True)
    y_batch = y_batch.to(device, non_blocking=True)
    
    optimizer.zero_grad()
    
    # Mixed Precision Forward Pass
    with torch.amp.autocast('cuda'):
        output = model(X_batch)
        loss = criterion(output, y_batch)
        
    # Mixed Precision Backward Pass
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    
    scheduler.step()
    
    batch_idx += 1
    progress_bar.update(1)
    progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})
    
    # Save Checkpoint
    if batch_idx % SAVE_INTERVAL == 0:
        torch.save({
            'batch_idx': batch_idx,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
        }, CHECKPOINT_PATH)
        
    if batch_idx >= TARGET_BATCHES:
        break

progress_bar.close()
# Final save
torch.save({
    'batch_idx': batch_idx,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scaler_state_dict': scaler.state_dict(),
    'scheduler_state_dict': scheduler.state_dict(),
}, CHECKPOINT_PATH)
print("Training target reached and saved!")


# 6. Export to .bin and Download
Exports the final quantized weights.

In [ ]:
import struct
from google.colab import files

def export_nnue(model, filename="nnue_weights.bin"):
    QA = 255
    QB = 64
    
    # Layer 1
    fc1_w = torch.round(model.fc1.weight.data * QA).clamp(-32768, 32767).to(torch.int16).flatten().tolist()
    fc1_b = torch.round(model.fc1.bias.data * QA).clamp(-32768, 32767).to(torch.int16).tolist()
    
    # Layer 2
    fc2_w = torch.round(model.fc2.weight.data * QB).clamp(-128, 127).to(torch.int8).flatten().tolist()
    fc2_b = torch.round(model.fc2.bias.data * (QA * QB)).to(torch.int32).tolist()
    
    # Layer 3
    fc3_w = torch.round(model.fc3.weight.data * QB).clamp(-128, 127).to(torch.int8).flatten().tolist()
    fc3_b = torch.round(model.fc3.bias.data * (QB * QB)).to(torch.int32).tolist()
    
    # Layer 4
    fc4_w = torch.round(model.fc4.weight.data * QB).clamp(-128, 127).to(torch.int8).flatten().tolist()
    fc4_b = torch.round(model.fc4.bias.data * (QB * QB)).to(torch.int32).tolist()

    with open(filename, 'wb') as f:
        f.write(struct.pack(f'<{len(fc1_w)}h', *fc1_w))
        f.write(struct.pack(f'<{len(fc1_b)}h', *fc1_b))
        
        f.write(struct.pack(f'<{len(fc2_w)}b', *fc2_w))
        f.write(struct.pack(f'<{len(fc2_b)}i', *fc2_b))
        
        f.write(struct.pack(f'<{len(fc3_w)}b', *fc3_w))
        f.write(struct.pack(f'<{len(fc3_b)}i', *fc3_b))
        
        f.write(struct.pack(f'<{len(fc4_w)}b', *fc4_w))
        f.write(struct.pack(f'<{len(fc4_b)}i', *fc4_b))
        
    print(f"Exported to {filename} ({len(fc1_w)*2 + len(fc1_b)*2 + len(fc2_w) + len(fc2_b)*4 + len(fc3_w) + len(fc3_b)*4 + len(fc4_w) + len(fc4_b)*4} bytes)")

# Export from CPU version to be safe
model.cpu()
export_nnue(model, "nnue_weights.bin")
files.download("nnue_weights.bin")
